# 03 — Validate susceptibility reconciliation

## Goal

The raw data can contain more than one result for the same culture-organism-antibiotic endpoint. I do not want to silently pick a row when those results disagree.

`int_susceptibility_reconciliation` classifies each endpoint as `resolved`, `missing`, or `conflict`. Only resolved endpoints are promoted to `fct_susceptibility`; the exceptions stay available for audit.

In [1]:
%sql
select
    resolution_status,
    count(*) as susceptibility_groups,
    sum(source_row_count) as source_rows
from workspace.micro_dev.int_susceptibility_reconciliation
group by resolution_status
order by resolution_status;

,resolution_status,susceptibility_groups,source_rows
0,conflict,9047,18424
1,missing,1991,1991
2,resolved,1588327,1588327


### Reconciliation outcomes

Most endpoints resolve cleanly: 1,588,327 groups are unambiguous. There are 9,047 conflicting groups and 1,991 missing-result groups.

The important design choice is that conflicts stay conflicts. I am not using a "latest result" or "most resistant result" rule to force them into the analytical fact.

In [2]:
%sql
select
    count(*) as susceptibility_rows,
    count(distinct susceptibility_key) as distinct_susceptibility_keys
from workspace.micro_dev.fct_susceptibility;

,susceptibility_rows,distinct_susceptibility_keys
0,1588327,1588327


### Fact-table uniqueness

The resolved susceptibility fact has 1,588,327 rows and the same number of unique susceptibility keys.

In [3]:
%sql
select
    culture_organism_key,
    antibiotic,
    count(*) as result_count
from workspace.micro_dev.fct_susceptibility
group by
    culture_organism_key,
    antibiotic
having count(*) > 1
order by result_count desc;

,culture_organism_key,antibiotic,result_count


### Duplicate endpoint check

This should return no rows. A duplicate here would mean the analytical fact no longer has one row per culture-organism-antibiotic endpoint.

In [4]:
%sql
select
    susceptibility,
    count(*) as result_count
from workspace.micro_dev.fct_susceptibility
group by susceptibility
order by result_count desc;

,susceptibility,result_count
0,Susceptible,1282464
1,Resistant,258619
2,Intermediate,43902
3,Synergism,1932
4,Inconclusive,1410


### Retained result categories

The fact keeps the source interpretation categories that are analytically usable, including `Inconclusive` and `Synergism`. Those are preserved for QA and coverage calculations even though they are not part of the S/R/I denominator.

In [5]:
%sql
select
    resolution_status,
    count(*) as susceptibility_groups,
    sum(source_row_count) as source_rows
from workspace.micro_dev.int_susceptibility_exceptions
group by resolution_status
order by resolution_status;

,resolution_status,susceptibility_groups,source_rows
0,conflict,9047,18424
1,missing,1991,1991


### Exception volume

The exception model is the audit trail for the records that were not promoted to the fact table. I keep missing and conflicting groups separate so they can be counted explicitly later.

In [6]:
%sql
select
    resolution_status,
    organism,
    antibiotic,
    distinct_result_state_count,
    distinct_nonnull_category_count,
    missing_susceptibility_row_count,
    repeated_result_row_count,
    source_row_count
from workspace.micro_dev.int_susceptibility_exceptions
order by
    distinct_result_state_count desc,
    source_row_count desc
limit 50;

,resolution_status,organism,antibiotic,distinct_result_state_count,distinct_nonnull_category_count,missing_susceptibility_row_count,repeated_result_row_count,source_row_count
0,conflict,PSEUDOMONAS AERUGINOSA (NON-MUCOID CF),Tobramycin,4,4,0,0,4
1,conflict,PSEUDOMONAS AERUGINOSA,Tobramycin,4,4,0,0,4
2,conflict,PSEUDOMONAS AERUGINOSA (NON-MUCOID CF),Tobramycin,4,4,0,0,4
3,conflict,PSEUDOMONAS AERUGINOSA (NON-MUCOID CF),Tobramycin,4,4,0,0,4
4,conflict,PSEUDOMONAS AERUGINOSA,Tobramycin,4,4,0,0,4
5,conflict,STREPTOCOCCUS PNEUMONIAE,Penicillin,4,3,1,0,4
6,conflict,PSEUDOMONAS AERUGINOSA (NON-MUCOID CF),Tobramycin,3,3,0,0,3
7,conflict,PSEUDOMONAS AERUGINOSA,Tobramycin,3,3,0,0,3
8,conflict,PSEUDOMONAS AERUGINOSA,Tobramycin,3,3,0,0,3
9,conflict,STREPTOCOCCUS PNEUMONIAE,Penicillin,3,3,0,0,3


### High-complexity conflicts

This pulls the messier exception groups so I can see what the source actually looks like. Some endpoints carry several distinct result states, which reinforces why an arbitrary winner would be hard to defend.

In [7]:
%sql
select
    organism,
    antibiotic,
    count(*) as conflicting_groups
from workspace.micro_dev.int_susceptibility_exceptions
where resolution_status = 'conflict'
group by
    organism,
    antibiotic
order by conflicting_groups desc
limit 50;

,organism,antibiotic,conflicting_groups
0,ESCHERICHIA COLI,Cefazolin,470
1,PSEUDOMONAS AERUGINOSA (NON-MUCOID CF),Tobramycin,378
2,ESCHERICHIA COLI,Ampicillin,333
3,PSEUDOMONAS AERUGINOSA,Tobramycin,309
4,ESCHERICHIA COLI,Trimethoprim/Sulfamethoxazole,276
5,MUCOID PSEUDOMONAS AERUGINOSA,Tobramycin,266
6,ESCHERICHIA COLI,Amoxicillin/Clavulanic Acid,220
7,STREPTOCOCCUS PNEUMONIAE,Penicillin,202
8,PSEUDOMONAS AERUGINOSA,Ciprofloxacin,190
9,ESCHERICHIA COLI,Ciprofloxacin,185


### Where conflicts are concentrated

The conflict profile is useful for understanding which organism-antibiotic combinations create the most reconciliation work. It also gives me something concrete to monitor if I update the source release later.

### Takeaway

The susceptibility layer now has a clear contract: resolved endpoints feed analysis, while ambiguous source states remain visible and countable.